In [102]:
import pandas as pd
import os

DATA_PATH = "../Datanad/subset_data"
df = pd.read_csv(os.path.join(DATA_PATH, "new_CGJ_084_-_IMS_202225.csv"), sep=";", encoding="utf-8-sig")

print("Colonnes disponibles :")
print(df.columns.tolist())

Colonnes disponibles :
['Nda', 'Date de naissance du patient', 'Date entrée UG entrée séjour', 'Date sortie UG entrée séjour', 'Nom du patient', 'Prénom du patient', 'Adresse du patient (rue)', 'Code postal de la ville', 'Code commune de la ville', 'Libelle de la ville', 'Libellé du Hameau/Lieu-dit', 'Identifiant de la categorie professionnelle', 'Libelle de la ville de naissance', 'Pays Naissance', 'UG entrée séjour Code', 'Mode sortie séjour Libellé', 'Décision urgence Libellé', 'Année', 'Nationalité (Patient)', 'Date Sortie Venue', 'Sejour Mode Sortie Libelle', 'Sejour Mode Sortie Pmsi', 'Urgence Date Sortie Completee']


/tmp/ipykernel_3939885/1964832843.py:5: DtypeWarning: Columns (8,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_PATH, "new_CGJ_084_-_IMS_202225.csv"), sep=";", encoding="utf-8-sig")


In [103]:
df["date_entree"] = pd.to_datetime(df["Date entrée UG entrée séjour"],
                                    format="%Y/%m/%d %H:%M:%S",
                                    errors="coerce")
df["annee_entree"] = df["date_entree"].dt.year

print(f"Total lignes : {len(df):,}")
print(f"Patients uniques (NDA) : {df['Nda'].nunique():,}")

print("\n=== Patients uniques par année d'entrée ===")
print(df.groupby("annee_entree")["Nda"].nunique().sort_index().to_string())

print(f"\nDates non parsées : {df['date_entree'].isna().sum():,}")

Total lignes : 341,561
Patients uniques (NDA) : 189,417

=== Patients uniques par année d'entrée ===
annee_entree
2022    48396
2023    43314
2024    46793
2025    50914

Dates non parsées : 0


In [104]:
# --- Filtre UG entrée séjour Code = 9780 ---
df_9780 = df[df["UG entrée séjour Code"] == 9780].copy()

print(f"Total lignes UG 9780 : {len(df_9780):,}")
print(f"Patients uniques UG 9780 : {df_9780['Nda'].nunique():,}")

print("\n=== Patients uniques UG 9780 par année ===")
df_9780["date_entree"] = pd.to_datetime(df_9780["Date entrée UG entrée séjour"],
                                         format="%Y/%m/%d %H:%M:%S", errors="coerce")
df_9780["annee_entree"] = df_9780["date_entree"].dt.year
print(df_9780.groupby("annee_entree")["Nda"].nunique().sort_index().to_string())

Total lignes UG 9780 : 337,265
Patients uniques UG 9780 : 187,001

=== Patients uniques UG 9780 par année ===
annee_entree
2022    47970
2023    42555
2024    46167
2025    50309


In [105]:
# --- Distribution de 'Décision urgence Libellé' par NDA unique ---

# On déduplique sur NDA en gardant la première occurrence
df_unique = df_9780.drop_duplicates(subset="Nda", keep="first")

print(f"Patients uniques : {len(df_unique):,}")
print(f"\n=== Décision urgence Libellé (% sur NDA uniques) ===")
print(
    df_unique["Décision urgence Libellé"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(1)
    .to_string()
)

Patients uniques : 187,001

=== Décision urgence Libellé (% sur NDA uniques) ===
Décision urgence Libellé
Consultation       52.5
Hospitalisation    45.4
NaN                 1.9
Pas de décision     0.2


In [111]:
# --- Filtre sur 'Consultation' ---
df_consult = df_unique[df_unique["Décision urgence Libellé"] == "Consultation"].copy()

print(f"Lignes avec 'Consultation' : {len(df_consult):,}")
print(f"NDA uniques 'Consultation' : {df_consult['Nda'].nunique():,}")

# --- Parser les deux dates ---
df_consult["date_sortie_venue"] = pd.to_datetime(df_consult["Date Sortie Venue"],
                                                  format="%Y/%m/%d %H:%M:%S", errors="coerce")
df_consult["date_sortie_complete"] = pd.to_datetime(df_consult["Urgence Date Sortie Completee"],
                                                     format="%Y/%m/%d %H:%M:%S", errors="coerce")
df_consult["Date sortie UG entrée séjour"] = pd.to_datetime(df_consult["Date sortie UG entrée séjour"],
                                                     format="%Y/%m/%d %H:%M:%S", errors="coerce")


# --- Comparer les dates (jour seulement, sans l'heure) ---
diff_mask = (
    df_consult["date_sortie_venue"].dt.date != df_consult["date_sortie_complete"].dt.date
) & df_consult["date_sortie_venue"].notna() & df_consult["date_sortie_complete"].notna()

print(f"\nLignes avec dates différentes : {diff_mask.sum():,}")
print(f"NDA uniques avec dates différentes : {df_consult[diff_mask]['Nda'].nunique():,}")

# --- Exemple ---
print("\n=== Exemples de dates différentes ===")
print(df_consult[diff_mask][["Nda", "Date Sortie Venue", "Urgence Date Sortie Completee"]].head(10).to_string())

Lignes avec 'Consultation' : 98,269
NDA uniques 'Consultation' : 98,269

Lignes avec dates différentes : 284
NDA uniques avec dates différentes : 284

=== Exemples de dates différentes ===
                Nda    Date Sortie Venue Urgence Date Sortie Completee
50547   23031209567  2023/12/29 14:26:00           2023/12/20 03:43:00
63470   22030852958  2022/11/30 13:27:00           2022/09/12 17:38:00
69749   24030527134  2024/07/01 17:42:00           2024/05/23 12:51:00
92970   25030738735  2025/07/16 17:06:00           2025/07/14 03:20:00
96542   25030606914  2025/06/09 14:04:00           2025/06/08 19:24:00
101802  24030343807  2024/05/16 14:52:00           2024/03/29 01:41:00
106883  22030803547  2022/08/27 06:45:00           2022/08/26 21:36:00
107818  22030789288  2022/08/24 04:45:00           2022/08/23 23:30:00
109105  22031088252  2022/11/24 13:21:00           2022/11/20 00:00:00
109321  22030345187  2022/04/16 12:30:00           2022/04/05 12:24:00


In [112]:
df_consult_unique = df_consult.drop_duplicates(subset="Nda", keep="first")

diff_mask_unique = (
    df_consult_unique["date_sortie_venue"].dt.date != df_consult_unique["date_sortie_complete"].dt.date
) & df_consult_unique["date_sortie_venue"].notna() & df_consult_unique["date_sortie_complete"].notna()

print(f"NDA uniques 'Consultation' : {len(df_consult_unique):,}")
print(f"NDA uniques avec dates différentes : {diff_mask_unique.sum():,}")
print(f"Pourcentage : {diff_mask_unique.sum() / len(df_consult_unique) * 100:.1f}%")

print("\n=== Exemples ===")
print(df_consult_unique[diff_mask_unique][["Nda", "Date Sortie Venue", "Urgence Date Sortie Completee"]].head(10).to_string())

df_consult_unique["diff_heures"] = (
    df_consult_unique["date_sortie_venue"] - df_consult_unique["date_sortie_complete"]
).dt.total_seconds() / 3600

diff_values = df_consult_unique.loc[diff_mask_unique, "diff_heures"]

print("\n=== Stats de la différence (en heures) ===")
print(f"Min    : {diff_values.min():.1f}h")
print(f"Max    : {diff_values.max():.1f}h")
print(f"Median : {diff_values.median():.1f}h")
print(f"Mean   : {diff_values.mean():.1f}h")
print(f"Std    : {diff_values.std():.1f}h")

# Distribution par tranches
print("\n=== Distribution par tranches ===")
bins = [-float('inf'), -24, -1, 0, 1, 24, 72, 168, float('inf')]
labels = ["< -24h", "-24h à -1h", "-1h à 0h", "0h à 1h", "1h à 24h", "24h à 72h", "72h à 1 semaine", "> 1 semaine"]
print(pd.cut(diff_values, bins=bins, labels=labels).value_counts().sort_index().to_string())





NDA uniques 'Consultation' : 98,269
NDA uniques avec dates différentes : 284
Pourcentage : 0.3%

=== Exemples ===
                Nda    Date Sortie Venue Urgence Date Sortie Completee
50547   23031209567  2023/12/29 14:26:00           2023/12/20 03:43:00
63470   22030852958  2022/11/30 13:27:00           2022/09/12 17:38:00
69749   24030527134  2024/07/01 17:42:00           2024/05/23 12:51:00
92970   25030738735  2025/07/16 17:06:00           2025/07/14 03:20:00
96542   25030606914  2025/06/09 14:04:00           2025/06/08 19:24:00
101802  24030343807  2024/05/16 14:52:00           2024/03/29 01:41:00
106883  22030803547  2022/08/27 06:45:00           2022/08/26 21:36:00
107818  22030789288  2022/08/24 04:45:00           2022/08/23 23:30:00
109105  22031088252  2022/11/24 13:21:00           2022/11/20 00:00:00
109321  22030345187  2022/04/16 12:30:00           2022/04/05 12:24:00

=== Stats de la différence (en heures) ===
Min    : 0.6h
Max    : 2338.9h
Median : 75.6h
Mean   : 145.6h

In [113]:
df_consult_unique["Date sortie UG entrée séjour2"] = pd.to_datetime(
    df_consult_unique["Date sortie UG entrée séjour"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)

df_consult_unique["date_sortie_venue"] = pd.to_datetime(
    df_consult_unique["Date Sortie Venue"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)

diff_mask_unique = (
    df_consult_unique["date_sortie_venue"].dt.date != df_consult_unique["Date sortie UG entrée séjour2"].dt.date
) & df_consult_unique["date_sortie_venue"].notna() & df_consult_unique["Date sortie UG entrée séjour2"].notna()

print(f"NDA uniques 'Consultation' : {len(df_consult_unique):,}")
print(f"NDA uniques avec dates différentes : {diff_mask_unique.sum():,}")
print(f"Pourcentage : {diff_mask_unique.sum() / len(df_consult_unique) * 100:.1f}%")

df_consult_unique["diff_heures"] = (
    df_consult_unique["date_sortie_venue"] - df_consult_unique["Date sortie UG entrée séjour2"]
).dt.total_seconds() / 3600

diff_values = df_consult_unique.loc[diff_mask_unique, "diff_heures"]

print("\n=== Stats de la différence Date Sortie Venue - Urgence Date Sortie Completee (heures) ===")
print(f"N      : {diff_values.notna().sum():,}")
print(f"Min    : {diff_values.min():.1f}h")
print(f"Max    : {diff_values.max():.1f}h")
print(f"Median : {diff_values.median():.1f}h")
print(f"Mean   : {diff_values.mean():.1f}h")
print(f"Std    : {diff_values.std():.1f}h")

# Distribution par tranches
print("\n=== Distribution par tranches ===")
bins = [-float('inf'), -24, -1, 0, 1, 12, 24, 72, 168, float('inf')]
labels = ["< -24h", "-24h à -1h", "-1h à 0h", "0h à 1h", "1h à 12", "12 à 24h", "24h à 72h", "3j a 1 semaine", " > 1semaine"]
print(pd.cut(diff_values, bins=bins, labels=labels).value_counts().sort_index().to_string())

NDA uniques 'Consultation' : 98,269
NDA uniques avec dates différentes : 1,005
Pourcentage : 1.0%

=== Stats de la différence Date Sortie Venue - Urgence Date Sortie Completee (heures) ===
N      : 1,005
Min    : 0.6h
Max    : 7158.3h
Median : 38.5h
Mean   : 135.1h
Std    : 347.1h

=== Distribution par tranches ===
diff_heures
< -24h              0
-24h à -1h          0
-1h à 0h            0
0h à 1h             2
1h à 12           199
12 à 24h          188
24h à 72h         240
3j a 1 semaine    195
 > 1semaine       181


In [114]:
# --- Patients avec dates différentes (l'une OU l'autre comparaison) ---

mask1 = (
    df_consult_unique["date_sortie_venue"].dt.date != df_consult_unique["Date sortie UG entrée séjour2"].dt.date
) & df_consult_unique["date_sortie_venue"].notna() & df_consult_unique["Date sortie UG entrée séjour2"].notna()

mask2 = (
    df_consult_unique["date_sortie_venue"].dt.date != df_consult_unique["date_sortie_complete"].dt.date
) & df_consult_unique["date_sortie_venue"].notna() & df_consult_unique["date_sortie_complete"].notna()

mask_either = mask1 | mask2

df_export = df_consult_unique[mask_either].copy()

df_export["diff_mask1_venue_VS_UGsejour"] = mask1[mask_either].values
df_export["diff_mask2_venue_VS_sortiecomplete"] = mask2[mask_either].values

print(f"NDA avec au moins une date différente : {len(df_export):,}")

# --- Colonnes à exclure ---
# --- Colonnes à exclure ---
cols_to_drop = [
    "Nom du patient", "Prénom du patient", "Adresse du patient (rue)",
    "Date de naissance du patient", "Année", "Annee",
    "Code postal de la ville", "Code commune de la ville",
    "Libelle de la ville", "Libellé du Hameau/Lieu-dit",
    "Identifiant de la categorie professionnelle",
    "Libelle de la ville de naissance", "Pays Naissance",
    "annee_entree", "Nationalité (Patient)",
    "Date sortie UG entrée séjour2", "date_entree",
    "date_sortie_venue", "date_sortie_complete", "diff_heures"
]
df_export = df_export.drop(columns=[c for c in cols_to_drop if c in df_export.columns])

# --- Colonnes fixes en tête ---
head_cols = ["Nda", "UG entrée séjour Code", "Date entrée UG entrée séjour"]

# --- Autres dates ---
date_cols = [c for c in df_export.columns if any(
    keyword in c.lower() for keyword in ["date", "heure"]
) and c not in head_cols]

# --- Reste ---
other_cols = [c for c in df_export.columns if c not in head_cols + date_cols]

final_order = head_cols + date_cols + other_cols
df_export = df_export[[c for c in final_order if c in df_export.columns]]

# --- Tri par NDA croissant ---
df_export = df_export.sort_values("Nda", ascending=True).reset_index(drop=True)

print("Colonnes finales :")
print(df_export.columns.tolist())

df_export.to_csv("patients_consultation_dates_differentes.csv", index=False, sep=";", encoding="utf-8-sig")
print(f"\n✅ Export : {len(df_export):,} patients → patients_consultation_dates_differentes.csv")

NDA avec au moins une date différente : 1,008
Colonnes finales :
['Nda', 'UG entrée séjour Code', 'Date entrée UG entrée séjour', 'Date sortie UG entrée séjour', 'Date Sortie Venue', 'Urgence Date Sortie Completee', 'Mode sortie séjour Libellé', 'Décision urgence Libellé', 'Sejour Mode Sortie Libelle', 'Sejour Mode Sortie Pmsi', 'diff_mask1_venue_VS_UGsejour', 'diff_mask2_venue_VS_sortiecomplete']

✅ Export : 1,008 patients → patients_consultation_dates_differentes.csv
